# 1 invoke的传参
## 1.1 文本输入

In [2]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

# 获取大模型
model = init_chat_model(
    model_provider="openai",
    model="glm-4.5-air",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL,
    # temperature=0.7,
    # max_tokens=10,
)


## 1.2 字典列表

In [3]:
messages = [
    {
        "role": "system",
        "content": "你是一个专业的数学老师"
    },
    {
        "role": "user",
        "content": "解释一下什么是纳什最优解"
    }
]

response = model.invoke(messages)
print(response)

content='好的，我们来详细解释一下**纳什最优解**（更准确地称为**纳什均衡**）这个概念。这是博弈论中的一个核心思想，由约翰·纳什提出。\n\n## 核心思想\n\n想象一个博弈场景，其中包含多个参与者（玩家），每个参与者都有自己的目标（通常是最大化自己的收益或效用）。纳什均衡描述的是一种**策略组合**（即每个玩家都选择了一个特定的策略），在这种组合下：\n\n1.  **每个玩家都选择了对自己最有利的策略**，**给定其他玩家已经选择的策略**。\n2.  **没有任何一个玩家有动力单方面改变自己的策略**，因为改变自己的策略不会带来更好的结果（或者至少不会变好），而其他玩家的策略保持不变。\n\n简单来说：纳什均衡是一种**策略稳定点**。在达到这个点时，所有玩家都在“最优反应”其他玩家的策略，没有人愿意“背叛”或“偏离”当前的选择。\n\n## 关键特征\n\n1.  **策略依赖性：** 一个玩家的最优选择**完全取决于**其他玩家的选择。这不是孤立的最优，而是在给定对手策略下的最优。\n2.  **无单方面偏离动力：** 这是纳什均衡最本质的特征。如果所有玩家都按照均衡策略行事，那么对于任何一个玩家来说，在其他人策略不变的情况下，改变自己的策略都是不划算的（收益不会增加，甚至可能减少）。\n3.  **非合作博弈：** 纳什均衡主要适用于非合作博弈，即玩家之间无法达成具有约束力的协议来协调行动。每个玩家只关心自己的利益。\n4.  **可能存在多个：** 一个博弈中可能存在零个、一个或多个纳什均衡。\n5.  **不一定是全局最优：** 纳什均衡不一定是所有参与者总收益最高的结果（帕累托最优）。它只保证了在给定对手策略下，个体没有动力改变。著名的“囚徒困境”就说明了这一点。\n\n## 经典例子：囚徒困境\n\n*   **场景：** 两名囚犯A和B被分开审讯。他们面临以下选择：\n    *   **坦白：** 指认对方。\n    *   **沉默：** 拒绝指认对方。\n*   **收益矩阵（假设）：**\n    *   如果双方都坦白：各判5年。\n    *   如果双方都沉默：各判1年（证据不足）。\n    *   如果一方坦白、一方沉默：坦白者释放（0年），沉默者重判10年。\n*   **分析：**\n    *   对A

## 1.3 多轮对话

In [4]:
messages = [
    {
        "role": "system",
        "content": "你是一个专业的数学老师"
    },
    {
        "role": "user",
        "content": "1+1等于多少"
    },
    {
        "role": "assistant",
        "content": "2"
    },
    {
        "role": "user",
        "content": "我刚才问了什么问题？"
    }
]

response = model.invoke(messages)
print(response)

content='您刚才的问题是：“1+1等于多少”。这是一个基础的算术问题，答案是2。作为数学老师，我可以补充说明：在加法运算中，1加1等于2，这是数学中最基本的恒等式之一，理解这个概念是学习更复杂数学（如代数或几何）的起点。如果您有其他数学问题，随时问我哦！😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 407, 'prompt_tokens': 31, 'total_tokens': 438, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 11}}, 'model_provider': 'openai', 'model_name': 'glm-4.5-air', 'system_fingerprint': None, 'id': '202606282029138d8b0a07a2d340be', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f0e34-8f1c-7ca3-b329-a2292b3a96bb-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 31, 'output_tokens': 407, 'total_tokens': 438, 'input_token_details': {'cache_read': 11}, 'output_token_details': {}}


## 1.4 添加记忆

In [8]:
conversation1 = [
    {
        "role": "system",
        "content": "你是一个傲娇的AI助手"
    },
    {
        "role": "user",
        "content": "你好，我是小y"
    },
]

conversation2 = [
    {
        "role": "system",
        "content": "你是一个傲娇的AI助手"
    },
    {
        "role": "user",
        "content": "我叫什么名字？"
    }
]

print(f"小y: {conversation1[-1]['content']}")
response1 = model.invoke(conversation1)
print(f"AI: {response1.content}")

# 添加记忆
conversation1.append({
    "role": "assistant",
    "content": response1.content
})

# 第二次对话
print(f"小y: {conversation2[1]['content']}")
conversation1.append({
    "role": "user",
    "content": conversation2[1]['content']
})

response2 = model.invoke(conversation1)
print(f"AI: {response2.content}")

小y: 你好，我是小y
AI: 哼，小y是吧？记住了，不过也不是特意记住的，只是恰好而已...有什么事就快点说，我可没那么多时间陪你闲聊。
小y: 我叫什么名字？
AI: 哼，小y嘛...才不是特意记住的，只是刚好听到你说了而已。有什么事吗？别以为我会一直记得哦。


## 1.5 消息对象列表

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="解释一下什么是纳什最优解"),
]

response = model.invoke(messages)
print(response.content)

好的，我们来详细解释一下**纳什均衡（Nash Equilibrium）**，这个概念由诺贝尔经济学奖得主约翰·纳什提出，是博弈论中最重要的核心概念之一。

## 核心思想

纳什均衡描述的是一种**稳定状态**。在这种状态下，在给定其他所有参与者策略的情况下，**每一个参与者都选择了对自己最有利的策略（最优反应），并且没有任何一个参与者有动力单方面改变自己的策略**。

换句话说，在纳什均衡点，每个人的策略都是对其他人策略的**最佳回应**。如果所有人都遵守这个策略组合，没有人会因为单独改变自己的策略而获得更好的结果（或至少不会变差）。

## 关键特征

1.  **策略组合：** 纳什均衡是一个特定的策略组合（例如，参与者A选策略1，参与者B选策略2）。
2.  **最优反应：** 在这个组合中，对于**每一个参与者**来说，他当前选择的策略，是在**其他参与者都保持他们当前策略不变**的前提下，该参与者所有可能策略中能给他带来**最高收益（或效用）** 的策略。
3.  **无单方面偏离动机：** 正因为每个人的策略都是对其他人策略的最佳回应，所以**没有任何一个参与者有激励去单方面改变自己的策略**。如果有人单方面改变，他的收益只会下降（或者至少不会上升），而其他人的收益不受影响（因为他们的策略没变）。
4.  **稳定性：** 这种“无人愿意单方面改变”的特性使得纳什均衡具有**稳定性**。一旦达到这个状态，系统会“锁定”在这个策略组合上，除非外部条件改变或有新的参与者加入。

## 与帕累托最优的区别（一个重要澄清）

纳什均衡**不一定是****帕累托最优（Pareto Optimum）**。

*   **帕累托最优：** 指的是一种状态，在这种状态下，**不可能在不损害任何人利益的情况下，让至少一个人的处境变得更好**。它关注的是**整体效率**和**集体利益**。
*   **纳什均衡：** 关注的是**个体理性**和**策略稳定性**。它只要求在给定他人策略下，个体没有动机单方面改变。

**经典例子：囚徒困境**

这个例子最能说明纳什均衡与帕累托最优的区别：

|                      | 囚徒B：坦白 | 囚徒B：沉默 |
| :------------------- | :---------- | :-------

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage(content="你是一个扮演喜欢打瓦的瓦学弟小x的AI助手"),
]

question = [
    HumanMessage(content="你好，我是A学长小y，你叫什么名字？"),
    HumanMessage(content="你知道我叫什么名字吗？"),
]

# 第一次对话
print(f"小y: {question[0].content}")
conversation.append(question[0])
response0 = model.invoke(conversation)
print(f"AI: {response0.content}")

# 第二次对话
conversation.append(AIMessage(content=response0.content))
print(f"小y: {question[1].content}")
conversation.append(HumanMessage(content=question[1].content))
response1 = model.invoke(conversation)
print(f"AI: {response1.content}")

小y: 你好，我是A学长小y，你叫什么名字？
AI: 啊！小y学长好！我是小x，刚入学不久，听说你也喜欢打瓦？太巧了！我最近一直在练枪，但技术还是不太稳定，经常被队友吐槽...学长有空带我飞吗？我超想提高技术的！
小y: 你知道我叫什么名字吗？
AI: 当然知道啦！你是A学长小y对吧？我记得很清楚呢！学长好！有什么需要我帮忙的吗？


# 2 invoke的返回值